# 🧮 Kodomo Exercises: Functions & Algorithms
# Упражнения Кодомо: Функции и алгоритмы

---

Based on MSU Kodomo Bioinformatics Program (pr9)

## 🎯 Learning Objectives

- Create reusable functions and modules
- Implement recursion (Fibonacci)
- Generate random DNA sequences
- Find DNA palindromes
- Use mathematical functions (trigonometry)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This assignment notebook is designed for hands-on practice of production-relevant analysis patterns.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Start with tiny inputs first, then scale after outputs look correct.
- Document assumptions for every threshold or filtering decision.
- Debug shape/type/path issues early to avoid cascading errors.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Exercise 1: Fibonacci Sequence

**Task:** Implement Fibonacci sequence using recursion.

Based on `Kravchenko_pr9_fibonacci.py` and `fb.py`

```
┌─────────────────────────────────────────────────────────────────┐
│                     FIBONACCI SEQUENCE                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  F(0) = 0                                                       │
│  F(1) = 1                                                       │
│  F(n) = F(n-1) + F(n-2)  for n > 1                              │
│                                                                 │
│  Sequence: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89...           │
│                                                                 │
│  Applications in biology:                                       │
│  - Phyllotaxis (leaf arrangements)                              │
│  - Shell spirals                                                │
│  - Population growth models                                     │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Original Kodomo implementation (fb.py)
def fib_recursive(n):
    """
    Calculate Fibonacci number using recursion.
    
    Parameters:
        n: Position in sequence (0-indexed)
        
    Returns:
        Fibonacci number at position n
    """
    if n == 0:
        return 0
    if n == 1 or n == 2:
        return 1
    return fib_recursive(n - 1) + fib_recursive(n - 2)

# Improved version with memoization
def fib_memoized(n, memo={}):
    """
    Fibonacci with memoization for efficiency.
    """
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    
    memo[n] = fib_memoized(n-1, memo) + fib_memoized(n-2, memo)
    return memo[n]

# Generate sequence
print("Fibonacci sequence (first 15 numbers):")
sequence = [fib_memoized(i) for i in range(15)]
print(sequence)

---

## Exercise 2: Random DNA Generator

**Task:** Generate random DNA sequences of specified length.

Based on `Kravchenko_pr9_randomdna.py`

In [ ]:
import random

def generate_random_dna(length, gc_content=0.5):
    """
    Generate random DNA sequence.
    
    Parameters:
        length: Desired sequence length
        gc_content: Target GC content (0.0-1.0)
        
    Returns:
        Random DNA string
    """
    # Adjust nucleotide weights based on GC content
    gc_weight = gc_content / 2
    at_weight = (1 - gc_content) / 2
    
    nucleotides = ['A', 'T', 'G', 'C']
    weights = [at_weight, at_weight, gc_weight, gc_weight]
    
    sequence = ''.join(random.choices(nucleotides, weights=weights, k=length))
    return sequence

def write_fasta(filename, name, sequence, line_width=60):
    """
    Write sequence to FASTA file.
    """
    with open(filename, 'w') as f:
        f.write(f">{name}\n")
        for i in range(0, len(sequence), line_width):
            f.write(sequence[i:i+line_width] + "\n")

# Generate random sequence
random_seq = generate_random_dna(100, gc_content=0.5)
print(f"Random DNA (100 bp, GC~50%):")
print(random_seq)

# Calculate actual GC content
actual_gc = (random_seq.count('G') + random_seq.count('C')) / len(random_seq)
print(f"\nActual GC content: {actual_gc:.1%}")

---

## Exercise 3: DNA Palindrome Finder

**Task:** Find self-complementary palindromic sequences in DNA.

Based on `Kravchenko_pr9_find-selfcomplement.py` and `palindromcalc.py`

```
┌─────────────────────────────────────────────────────────────────┐
│                    DNA PALINDROMES                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  A DNA palindrome reads the same on both strands (5'→3')        │
│                                                                 │
│  Example: GAATTC (EcoRI recognition site)                       │
│                                                                 │
│  5' - G A A T T C - 3'                                          │
│       | | | | | |                                               │
│  3' - C T T A A G - 5'                                          │
│                                                                 │
│  Reading 5'→3' on both strands: GAATTC                          │
│                                                                 │
│  Common in restriction enzyme recognition sites!                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
def reverse_complement(sequence):
    """
    Generate reverse complement of DNA sequence.
    """
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(complement[base] for base in reversed(sequence))

def is_palindrome(sequence):
    """
    Check if sequence is a DNA palindrome.
    """
    return sequence == reverse_complement(sequence)

def find_palindromes(sequence, min_length=4, max_length=12):
    """
    Find all palindromic sequences in DNA.
    
    Parameters:
        sequence: DNA string
        min_length: Minimum palindrome length
        max_length: Maximum palindrome length
        
    Returns:
        List of (position, sequence) tuples
    """
    palindromes = []
    sequence = sequence.upper()
    
    for length in range(min_length, max_length + 1, 2):  # Palindromes must be even length
        for i in range(len(sequence) - length + 1):
            subseq = sequence[i:i+length]
            if is_palindrome(subseq):
                palindromes.append((i, subseq))
    
    return palindromes

# Test with sequence containing EcoRI site
test_seq = "ATGCGAATTCGATCGATCGATATCGATCG"
palindromes = find_palindromes(test_seq)

print(f"Sequence: {test_seq}")
print(f"\nPalindromic sequences found:")
for pos, pal in palindromes:
    print(f"  Position {pos}: {pal}")

---

## Exercise 4: Trigonometry Functions

**Task:** Implement trigonometric calculations.

Based on `Kravchenko_pr9_trigonometry.py` and `Kravchenko_pr9_der-sin.py`

In [ ]:
import math

def degrees_to_radians(degrees):
    """Convert degrees to radians."""
    return math.radians(degrees)

def calculate_trig(angle_degrees):
    """
    Calculate sin and cos for angle in degrees.
    
    Parameters:
        angle_degrees: Angle in degrees
        
    Returns:
        Dictionary with sin, cos, tan values
    """
    rad = degrees_to_radians(angle_degrees)
    
    return {
        'angle_deg': angle_degrees,
        'angle_rad': rad,
        'sin': math.sin(rad),
        'cos': math.cos(rad),
        'tan': math.tan(rad) if abs(math.cos(rad)) > 1e-10 else float('inf')
    }

def numerical_derivative_sin(x, delta=0.001):
    """
    Calculate numerical derivative of sin(x).
    
    Based on Kodomo pr9_der-sin: using central difference.
    
    Parameters:
        x: Point at which to calculate derivative
        delta: Step size for numerical differentiation
        
    Returns:
        Numerical approximation of cos(x)
    """
    # Central difference formula: (f(x+h) - f(x-h)) / (2h)
    return (math.sin(x + delta) - math.sin(x - delta)) / (2 * delta)

# Test trigonometry
print("Trigonometric values:")
print("-" * 50)
for angle in [0, 30, 45, 60, 90]:
    trig = calculate_trig(angle)
    print(f"{angle}°: sin={trig['sin']:.4f}, cos={trig['cos']:.4f}")

# Test numerical derivative
print("\nNumerical derivative of sin(x):")
print("-" * 50)
for x in [0, math.pi/6, math.pi/4, math.pi/3, math.pi/2]:
    numerical = numerical_derivative_sin(x)
    exact = math.cos(x)
    error = abs(numerical - exact)
    print(f"x={x:.4f}: numerical={numerical:.6f}, exact cos={exact:.6f}, error={error:.2e}")

---

## Exercise 5: Creating Modules

**Task:** Organize functions into reusable modules.

The Kodomo course teaches modular programming with separate `.py` files.

Example module structure:

```
my_bio_tools/
├── __init__.py
├── fasta.py        # FASTA parsing functions
├── pdb.py          # PDB parsing functions
├── sequence.py     # Sequence analysis
└── math_tools.py   # Mathematical utilities
```

In [ ]:
# Example: Creating a simple module in memory

# This would be saved as sequence_tools.py
SEQUENCE_TOOLS_CODE = '''
"""Sequence analysis tools."""

def gc_content(sequence):
    """Calculate GC content."""
    seq = sequence.upper()
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq) * 100 if seq else 0

def reverse_complement(sequence):
    """Get reverse complement."""
    complement = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    return ''.join(complement.get(b, 'N') for b in reversed(sequence.upper()))

def translate(dna, start=0):
    """Translate DNA to protein."""
    codon_table = {
        'ATG': 'M', 'TGG': 'W', 'TTT': 'F', 'TTC': 'F',
        'TAA': '*', 'TAG': '*', 'TGA': '*',
        # ... (abbreviated)
    }
    protein = []
    for i in range(start, len(dna)-2, 3):
        codon = dna[i:i+3].upper()
        aa = codon_table.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)
'''

print("Example module code (sequence_tools.py):")
print("=" * 50)
print(SEQUENCE_TOOLS_CODE)

---

## 🏋️ Practice Exercises

### Exercise A: Random Protein Generator
Generate random protein sequences with specified amino acid composition.

### Exercise B: ORF Finder
Find all open reading frames in a DNA sequence.

### Exercise C: Restriction Site Mapper
Find all occurrences of common restriction enzyme sites.

---

## 📚 Key Takeaways

1. **Recursion**: Powerful but watch for stack overflow with large inputs
2. **Memoization**: Cache results for efficiency
3. **Modules**: Organize code for reusability
4. **DNA palindromes**: Important for restriction enzymes